# Notebook 5: Hierarchical Pipeline (Experimental)

This notebook documents an **experimental hierarchical two-stage approach** to collision severity prediction.

**Approach:**
1. **Stage 1** — Binary: Fatal vs Non-Fatal (with SMOTE + focal loss + threshold calibration)
2. **Stage 2** — Binary: Serious vs Slight (for samples predicted Non-Fatal)

**Outcome:** Fatal Recall improved to ~80% (vs 58.8% in main pipeline), but Macro F1 dropped to ~0.23 (vs 0.31). The extreme class imbalance (76 real Fatal samples) limits Stage 1's reliability. This notebook is kept for completeness and to demonstrate the precision-recall trade-off.

**Best result from main pipeline (Notebook 4):** Fatal Recall=68.8%, Macro F1=0.311 (Step 8, architecture search + threshold).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import matthews_corrcoef, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
import joblib, copy, time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else
                       'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load data
data = joblib.load('../outputs/models/preprocessed_data.joblib')
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']
feature_names    = data['feature_names']
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

# ── Re-define shared utilities ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp().clamp(1e-7, 1.0 - 1e-7)
        loss = -(1.0 - pt) ** self.gamma * log_pt
        if self.alpha is not None:
            loss = self.alpha.gather(0, targets) * loss
        return loss.mean() if self.reduction == 'mean' else loss.sum()

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm1d):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

@torch.no_grad()
def evaluate_binary(model, loader, criterion, device, pos_label=1):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_targets, all_probs = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * xb.size(0)
        n += xb.size(0)
        probs = torch.softmax(logits, dim=-1)[:, pos_label].cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(yb.cpu().numpy())
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    return {
        'loss': total_loss / n,
        'mcc': matthews_corrcoef(all_targets, all_preds),
        'f1': f1_score(all_targets, all_preds, average='binary', pos_label=pos_label, zero_division=0),
        'macro_f1': f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'recall': recall_score(all_targets, all_preds, pos_label=pos_label, zero_division=0),
        'probs': all_probs, 'targets': all_targets,
    }

class BinaryMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

def predict_hierarchical(X, s1_model, s2_model, device, threshold_s1):
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    s1_model.eval()
    with torch.no_grad():
        s1_probs = torch.softmax(s1_model(X_tensor), dim=-1)[:, 1].cpu().numpy()
    fatal_preds = s1_probs >= threshold_s1
    final_preds = np.full(len(X), -1)
    final_preds[fatal_preds] = 0
    non_fatal_idx = np.where(~fatal_preds)[0]
    if len(non_fatal_idx) > 0:
        X_nf = torch.tensor(X[non_fatal_idx], dtype=torch.float32).to(device)
        s2_model.eval()
        with torch.no_grad():
            s2_preds = s2_model(X_nf).argmax(dim=1).cpu().numpy()
        final_preds[non_fatal_idx] = s2_preds + 1
    return final_preds

print('All utilities loaded.')

## Step 10: Feature Selection

The diagnostic revealed that 5 out of 21 features have **weak discriminative power** for Fatal crashes (Cohen's d < 0.12): `urban_or_rural_area`, `max_driver_age`, `first_road_class`, `day_of_week`, and `speed_limit`.

With only 76 Fatal training samples, every noisy dimension hurts the model's ability to learn the Fatal decision boundary. Removing these 5 features improves the samples-per-feature ratio from 3.6 to 4.75 and reduces noise.

In [16]:
# ================================================================
# Step 10: Feature Selection — Remove Weak Features
# ================================================================

# Features to drop (Cohen's d < 0.12 for Fatal vs Non-Fatal)
weak_features = ['urban_or_rural_area', 'max_driver_age', 'first_road_class', 'day_of_week', 'speed_limit']

# Get indices of features to keep
feature_names_list = list(feature_names)
keep_indices = [i for i, name in enumerate(feature_names_list) if name not in weak_features]
selected_features = [feature_names_list[i] for i in keep_indices]

print(f"Original features: {len(feature_names_list)}")
print(f"Dropped ({len(weak_features)}): {weak_features}")
print(f"Kept ({len(selected_features)}): {selected_features}")

# Apply selection to all splits
X_train_sel = X_train[:, keep_indices]
X_val_sel = X_val[:, keep_indices]
X_test_sel = X_test[:, keep_indices]

print(f"\nNew shapes: Train {X_train_sel.shape}, Val {X_val_sel.shape}, Test {X_test_sel.shape}")
print(f"Samples-per-feature (Fatal): {(y_train == 0).sum() / X_train_sel.shape[1]:.1f} (was 3.6)")

Original features: 21
Dropped (5): ['urban_or_rural_area', 'max_driver_age', 'first_road_class', 'day_of_week', 'speed_limit']
Kept (16): ['number_of_vehicles', 'number_of_casualties', 'road_type', 'junction_detail', 'junction_control', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'total_casualties', 'min_casualty_age', 'max_casualty_age', 'total_vehicles_involved', 'hour']

New shapes: Train (14676, 16), Val (3145, 16), Test (3146, 16)
Samples-per-feature (Fatal): 4.8 (was 3.6)


## Steps 11-13: Hierarchical Two-Stage Classification with SMOTE & Threshold Calibration

Instead of forcing one model to draw three decision boundaries simultaneously, we decompose the problem:

1. **Stage 1 — Fatal vs Non-Fatal** (binary): Uses SMOTE to generate synthetic Fatal samples, a focused binary MLP, and threshold calibration. All model capacity goes toward the life-or-death question.
2. **Stage 2 — Serious vs Slight** (binary): Only runs on samples predicted Non-Fatal in Stage 1. Much easier problem with more balanced data.

**SMOTE** (Step 12) generates new interpolated Fatal samples by blending between existing ones in feature space — unlike the WeightedRandomSampler which just repeats the same 76 examples.

**Threshold Calibration** (Step 13) tunes the Fatal decision threshold on validation data. Binary thresholds are much more effective than multiclass ones because we only have one number to optimise.

In [17]:
# ================================================================
# Steps 11-13: Hierarchical Two-Stage Classification
# ================================================================
from imblearn.over_sampling import SMOTE

# --- 11a. Prepare binary labels ---
# Stage 1: Fatal (1) vs Non-Fatal (0)
y_train_s1 = (y_train == 0).astype(np.int64)
y_val_s1 = (y_val == 0).astype(np.int64)
y_test_s1 = (y_test == 0).astype(np.int64)

# Stage 2: Serious (0) vs Slight (1) — only for non-fatal samples
nf_train = y_train != 0
nf_val = y_val != 0
nf_test = y_test != 0

# Stage 2 labels: remap Serious=1→0, Slight=2→1
y_train_s2 = (y_train[nf_train] - 1).astype(np.int64)
y_val_s2 = (y_val[nf_val] - 1).astype(np.int64)

print("Stage 1 — Fatal vs Non-Fatal:")
print(f"  Train: Fatal={y_train_s1.sum()}, Non-Fatal={(1-y_train_s1).sum()}")
print(f"  Val:   Fatal={y_val_s1.sum()}, Non-Fatal={(1-y_val_s1).sum()}")

print(f"\nStage 2 — Serious vs Slight (non-fatal only):")
print(f"  Train: Serious={(y_train_s2==0).sum()}, Slight={(y_train_s2==1).sum()}")
print(f"  Val:   Serious={(y_val_s2==0).sum()}, Slight={(y_val_s2==1).sum()}")

# --- 12a. Apply SMOTE to Stage 1 training data ---
print(f"\n{'='*60}")
print(f"  STEP 12: SMOTE FOR FATAL CLASS")
print(f"{'='*60}")

# Use selected features (16 features from Step 10)
smote = SMOTE(sampling_strategy=0.05, k_neighbors=3, random_state=SEED)
X_train_s1_smote, y_train_s1_smote = smote.fit_resample(X_train_sel, y_train_s1)

print(f"Before SMOTE: {X_train_sel.shape[0]} samples (Fatal={y_train_s1.sum()})")
print(f"After SMOTE:  {X_train_s1_smote.shape[0]} samples "
      f"(Fatal={y_train_s1_smote.sum()}, Non-Fatal={(1-y_train_s1_smote).sum()})")
print(f"Synthetic Fatal samples generated: {y_train_s1_smote.sum() - y_train_s1.sum()}")
print(f"New ratio: 1:{(1-y_train_s1_smote).sum() / y_train_s1_smote.sum():.0f}")

# --- 11b. Build Stage 1 model ---
class BinaryMLP(nn.Module):
    """Simple binary MLP for stage classification."""
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# --- 11c. Train Stage 1: Fatal vs Non-Fatal ---
print(f"\n{'='*60}")
print(f"  STAGE 1: TRAINING Fatal vs Non-Fatal")
print(f"{'='*60}")

# DataLoaders with WeightedRandomSampler
s1_tensor = TensorDataset(
    torch.tensor(X_train_s1_smote, dtype=torch.float32),
    torch.tensor(y_train_s1_smote, dtype=torch.long)
)
# No WeightedRandomSampler — SMOTE already balanced the training data.
# Adding a sampler on top would double-overcorrect and flag everything as Fatal.
s1_train_loader = DataLoader(s1_tensor, batch_size=64, shuffle=True)

s1_val_tensor = TensorDataset(
    torch.tensor(X_val_sel, dtype=torch.float32),
    torch.tensor(y_val_s1, dtype=torch.long)
)
s1_val_loader = DataLoader(s1_val_tensor, batch_size=256, shuffle=False)

# Binary Focal Loss with mild alpha for Fatal
s1_model = BinaryMLP(X_train_sel.shape[1], [64, 32], dropout=0.3)
s1_model.apply(init_weights)
s1_model = s1_model.to(device)

s1_criterion = FocalLoss(alpha=[1.0, 1.5], gamma=2.0).to(device)
s1_opt = optim.AdamW(s1_model.parameters(), lr=1e-3, weight_decay=1e-4)
s1_ws = torch.optim.lr_scheduler.LinearLR(s1_opt, start_factor=0.1, total_iters=5)
s1_cs = torch.optim.lr_scheduler.CosineAnnealingLR(s1_opt, T_max=95, eta_min=1e-5)
s1_sched = torch.optim.lr_scheduler.SequentialLR(s1_opt, schedulers=[s1_ws, s1_cs], milestones=[5])

# Custom evaluate for binary
@torch.no_grad()
def evaluate_binary(model, loader, criterion, device, pos_label=1):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_targets, all_probs = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * xb.size(0)
        n += xb.size(0)
        probs = torch.softmax(logits, dim=-1)[:, pos_label].cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(yb.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)
    
    recall_pos = recall_score(all_targets, all_preds, pos_label=pos_label, zero_division=0)
    return {
        'loss': total_loss / n,
        'mcc': matthews_corrcoef(all_targets, all_preds),
        'f1': f1_score(all_targets, all_preds, average='binary', pos_label=pos_label, zero_division=0),
        'macro_f1': f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'recall': recall_pos,
        'probs': all_probs,
        'targets': all_targets,
    }

# Training loop
best_f1 = -1
best_state_s1 = None
best_epoch_s1 = 0
patience_counter = 0
s1_history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_recall': [], 'val_mcc': []}

t0 = time.time()
for epoch in range(100):
    s1_model.train()
    rl, ns = 0.0, 0
    for xb, yb in s1_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        s1_opt.zero_grad()
        loss = s1_criterion(s1_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(s1_model.parameters(), max_norm=1.0)
        s1_opt.step()
        rl += loss.item() * xb.size(0)
        ns += xb.size(0)
    
    v = evaluate_binary(s1_model, s1_val_loader, s1_criterion, device, pos_label=1)
    s1_sched.step()
    
    s1_history['train_loss'].append(rl / ns)
    s1_history['val_loss'].append(v['loss'])
    s1_history['val_f1'].append(v['f1'])
    s1_history['val_recall'].append(v['recall'])
    s1_history['val_mcc'].append(v['mcc'])
    
    # Save best by F1 with recall gate
    if v['recall'] >= 0.20 and v['f1'] > best_f1:
        best_f1 = v['f1']
        best_state_s1 = copy.deepcopy(s1_model.state_dict())
        best_epoch_s1 = epoch
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= 25:
        break

s1_time = time.time() - t0
print(f"Stage 1 trained in {s1_time:.1f}s | Best epoch: {best_epoch_s1} | Epochs: {epoch+1}")

# Restore best
if best_state_s1 is not None:
    s1_model.load_state_dict(best_state_s1)
s1_model.eval()

v_final = evaluate_binary(s1_model, s1_val_loader, s1_criterion, device, pos_label=1)
print(f"Stage 1 (argmax): Fatal Recall={v_final['recall']:.4f} | F1={v_final['f1']:.4f} | "
      f"MCC={v_final['mcc']:.4f}")

# --- 13a. Binary Threshold Calibration for Stage 1 ---
print(f"\n{'='*60}")
print(f"  STEP 13: BINARY THRESHOLD CALIBRATION")
print(f"{'='*60}")

s1_probs = v_final['probs']  # P(Fatal) for each val sample
s1_targets = v_final['targets']

print(f"{'Threshold':>10} {'Fatal R':>8} {'Precision':>10} {'F1':>7} {'MCC':>7}")
print("-" * 50)

best_thresh_s1 = 0.5
best_recall_s1 = v_final['recall']
best_f1_s1 = v_final['f1']
best_mcc_s1 = v_final['mcc']

for thresh in np.arange(0.05, 0.60, 0.01):
    preds = (s1_probs >= thresh).astype(int)
    rec = recall_score(s1_targets, preds, pos_label=1, zero_division=0)
    prec = np.sum((preds == 1) & (s1_targets == 1)) / max(np.sum(preds == 1), 1)
    f1_val = 2 * prec * rec / max(prec + rec, 1e-8)
    mcc_val = matthews_corrcoef(s1_targets, preds)
    
    print(f"{thresh:10.2f} {rec:8.4f} {prec:10.4f} {f1_val:7.4f} {mcc_val:7.4f}")
    
    # Pick threshold that maximises Fatal Recall while keeping MCC > 0.05 (meaningful signal)
    if rec > best_recall_s1 and mcc_val > 0.05:
        best_thresh_s1 = thresh
        best_recall_s1 = rec
        best_f1_s1 = f1_val
        best_mcc_s1 = mcc_val

print(f"\nOptimal Stage 1 threshold: {best_thresh_s1:.2f}")
print(f"  Fatal Recall: {best_recall_s1:.4f}")
print(f"  Fatal F1: {best_f1_s1:.4f}")
print(f"  MCC: {best_mcc_s1:.4f}")

Stage 1 — Fatal vs Non-Fatal:
  Train: Fatal=76, Non-Fatal=14600
  Val:   Fatal=16, Non-Fatal=3129

Stage 2 — Serious vs Slight (non-fatal only):
  Train: Serious=2445, Slight=12155
  Val:   Serious=524, Slight=2605

  STEP 12: SMOTE FOR FATAL CLASS
Before SMOTE: 14676 samples (Fatal=76)
After SMOTE:  15330 samples (Fatal=730, Non-Fatal=14600)
Synthetic Fatal samples generated: 654
New ratio: 1:20

  STAGE 1: TRAINING Fatal vs Non-Fatal


Stage 1 trained in 33.0s | Best epoch: 0 | Epochs: 25
Stage 1 (argmax): Fatal Recall=0.0000 | F1=0.0000 | MCC=-0.0040

  STEP 13: BINARY THRESHOLD CALIBRATION
 Threshold  Fatal R  Precision      F1     MCC
--------------------------------------------------
      0.05   1.0000     0.0071  0.0141  0.0447
      0.06   1.0000     0.0072  0.0143  0.0462
      0.07   1.0000     0.0074  0.0147  0.0480
      0.08   1.0000     0.0075  0.0150  0.0497
      0.09   1.0000     0.0077  0.0153  0.0514
      0.10   1.0000     0.0080  0.0158  0.0540
      0.11   1.0000     0.0083  0.0165  0.0569
      0.12   0.9375     0.0083  0.0165  0.0525
      0.13   0.9375     0.0087  0.0173  0.0563
      0.14   0.9375     0.0092  0.0182  0.0598
      0.15   0.8125     0.0082  0.0163  0.0445
      0.16   0.8125     0.0086  0.0170  0.0475
      0.17   0.8125     0.0090  0.0178  0.0506
      0.18   0.8125     0.0094  0.0186  0.0537
      0.19   0.8125     0.0098  0.0194  0.0568
      0.20   0.7500     0.0095  0.0188

In [18]:
# ================================================================
# Stage 2: Serious vs Slight + Combined Pipeline Evaluation
# ================================================================

# --- Train Stage 2: Serious vs Slight ---
print(f"{'='*60}")
print(f"  STAGE 2: TRAINING Serious vs Slight")
print(f"{'='*60}")

X_train_s2_sel = X_train_sel[nf_train]
X_val_s2_sel = X_val_sel[nf_val]

s2_tensor = TensorDataset(
    torch.tensor(X_train_s2_sel, dtype=torch.float32),
    torch.tensor(y_train_s2, dtype=torch.long)
)
s2_counts = np.bincount(y_train_s2, minlength=2)
s2_weights = 1.0 / s2_counts[y_train_s2]
s2_sampler = WeightedRandomSampler(torch.tensor(s2_weights, dtype=torch.float64),
                                    num_samples=len(y_train_s2), replacement=True)
s2_train_loader = DataLoader(s2_tensor, batch_size=64, sampler=s2_sampler)

s2_val_tensor = TensorDataset(
    torch.tensor(X_val_s2_sel, dtype=torch.float32),
    torch.tensor(y_val_s2, dtype=torch.long)
)
s2_val_loader = DataLoader(s2_val_tensor, batch_size=256, shuffle=False)

s2_model = BinaryMLP(X_train_sel.shape[1], [64, 32], dropout=0.3)
s2_model.apply(init_weights)
s2_model = s2_model.to(device)

# Standard focal loss — Serious vs Slight is more balanced
s2_criterion = FocalLoss(alpha=[1.0, 1.0], gamma=1.0).to(device)
s2_opt = optim.AdamW(s2_model.parameters(), lr=1e-3, weight_decay=1e-4)
s2_ws = torch.optim.lr_scheduler.LinearLR(s2_opt, start_factor=0.1, total_iters=5)
s2_cs = torch.optim.lr_scheduler.CosineAnnealingLR(s2_opt, T_max=95, eta_min=1e-5)
s2_sched = torch.optim.lr_scheduler.SequentialLR(s2_opt, schedulers=[s2_ws, s2_cs], milestones=[5])

best_f1_s2 = -1
best_state_s2 = None
best_epoch_s2 = 0
patience_s2 = 0

t0 = time.time()
for epoch in range(100):
    s2_model.train()
    for xb, yb in s2_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        s2_opt.zero_grad()
        loss = s2_criterion(s2_model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(s2_model.parameters(), max_norm=1.0)
        s2_opt.step()
    
    v2 = evaluate_binary(s2_model, s2_val_loader, s2_criterion, device, pos_label=0)
    s2_sched.step()
    
    if v2['macro_f1'] > best_f1_s2:
        best_f1_s2 = v2['macro_f1']
        best_state_s2 = copy.deepcopy(s2_model.state_dict())
        best_epoch_s2 = epoch
        patience_s2 = 0
    else:
        patience_s2 += 1
    if patience_s2 >= 25:
        break

s2_time = time.time() - t0
if best_state_s2 is not None:
    s2_model.load_state_dict(best_state_s2)
s2_model.eval()

v2_final = evaluate_binary(s2_model, s2_val_loader, s2_criterion, device, pos_label=0)
print(f"Stage 2 trained in {s2_time:.1f}s | Best epoch: {best_epoch_s2}")
print(f"Stage 2: Serious Recall={v2_final['recall']:.4f} | Macro F1={v2_final['macro_f1']:.4f} | MCC={v2_final['mcc']:.4f}")

# ================================================================
# COMBINED PIPELINE — Full 3-Class Prediction
# ================================================================
print(f"\n{'='*60}")
print(f"  COMBINED HIERARCHICAL PIPELINE EVALUATION")
print(f"{'='*60}")

def predict_hierarchical(X, s1_model, s2_model, device, threshold_s1, input_sel=None):
    """Run the full 2-stage hierarchical pipeline."""
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    
    # Stage 1: Fatal vs Non-Fatal
    s1_model.eval()
    with torch.no_grad():
        s1_probs = torch.softmax(s1_model(X_tensor), dim=-1)[:, 1].cpu().numpy()
    
    # Apply calibrated threshold
    fatal_preds = s1_probs >= threshold_s1
    
    # Stage 2: Serious vs Slight (for non-fatal samples)
    final_preds = np.full(len(X), -1)
    final_preds[fatal_preds] = 0  # Fatal
    
    non_fatal_idx = np.where(~fatal_preds)[0]
    if len(non_fatal_idx) > 0:
        X_nf = torch.tensor(X[non_fatal_idx], dtype=torch.float32).to(device)
        s2_model.eval()
        with torch.no_grad():
            s2_logits = s2_model(X_nf)
            s2_preds = s2_logits.argmax(dim=1).cpu().numpy()
        
        # Map back: 0→Serious(1), 1→Slight(2)
        final_preds[non_fatal_idx] = s2_preds + 1
    
    return final_preds

# Evaluate on validation set
hier_preds = predict_hierarchical(X_val_sel, s1_model, s2_model, device, best_thresh_s1)

hier_recall = recall_score(y_val, hier_preds, average=None, labels=[0, 1, 2], zero_division=0)
hier_mcc = matthews_corrcoef(y_val, hier_preds)
hier_macro_f1 = f1_score(y_val, hier_preds, average='macro', zero_division=0)

print(f"\nHierarchical Pipeline (threshold={best_thresh_s1:.2f}):")
print(f"  Fatal Recall:   {hier_recall[0]:.4f}")
print(f"  Serious Recall: {hier_recall[1]:.4f}")
print(f"  Slight Recall:  {hier_recall[2]:.4f}")
print(f"  MCC:            {hier_mcc:.4f}")
print(f"  Macro F1:       {hier_macro_f1:.4f}")

# --- Comprehensive comparison ---
print(f"\n{'='*70}")
print(f"  FULL COMPARISON — ALL APPROACHES")
print(f"{'='*70}")
print(f"  {'Method':<35} {'Fatal R':>8} {'Serious R':>10} {'Slight R':>9} {'MCC':>7} {'Macro F1':>9}")
print(f"  {'-'*78}")
print(f"  {'RF Baseline':<35} {'0.0000':>8} {'0.2854':>10} {'0.9590':>9} {'0.0277':>7} {'0.3088':>9}")
print(f"  {'3-Class MLP (Step 5 winner)':<35} {'0.5000':>8} {'0.3550':>10} {'0.5727':>9} {'0.0987':>7} {'0.3289':>9}")

# Best from ensemble threshold
print(f"  {'Ensemble + Threshold (Step 7)':<35} {'0.6250':>8} {'0.3015':>10} {'0.5735':>9} {'0.1039':>7} {'0.3248':>9}")

# Hierarchical
print(f"  {'Hierarchical Pipeline (Steps 11-13)':<35} {hier_recall[0]:8.4f} {hier_recall[1]:10.4f} "
      f"{hier_recall[2]:9.4f} {hier_mcc:7.4f} {hier_macro_f1:9.4f}")
print(f"  {'-'*78}")

# Confusion matrix
cm = confusion_matrix(y_val, hier_preds, labels=[0, 1, 2])
print(f"\nConfusion Matrix (Hierarchical):")
print(f"  {'':>12} {'Pred Fatal':>11} {'Pred Serious':>13} {'Pred Slight':>12}")
for i, name in enumerate(['Fatal', 'Serious', 'Slight']):
    print(f"  {name:>12} {cm[i,0]:11d} {cm[i,1]:13d} {cm[i,2]:12d}")

# Save hierarchical model
torch.save({
    's1_state_dict': s1_model.state_dict(),
    's2_state_dict': s2_model.state_dict(),
    's1_threshold': best_thresh_s1,
    'architecture': {'input_dim': X_train_sel.shape[1], 'hidden_dims': [64, 32]},
    'selected_features': selected_features,
    'keep_indices': keep_indices,
    'val_metrics': {
        'fatal_recall': hier_recall[0],
        'serious_recall': hier_recall[1],
        'slight_recall': hier_recall[2],
        'mcc': hier_mcc,
        'macro_f1': hier_macro_f1,
    },
}, '../outputs/models/hierarchical_model.pt')
print(f"\nHierarchical model saved to outputs/models/hierarchical_model.pt")

# Save all improvement results
joblib.dump({
    's1_history': s1_history,
    'best_thresh_s1': best_thresh_s1,
    'hier_val_metrics': {
        'fatal_recall': float(hier_recall[0]),
        'serious_recall': float(hier_recall[1]),
        'slight_recall': float(hier_recall[2]),
        'mcc': float(hier_mcc),
        'macro_f1': float(hier_macro_f1),
    },
    'confusion_matrix': cm.tolist(),
}, '../outputs/models/hierarchical_results.joblib')
print(f"Results saved to outputs/models/hierarchical_results.joblib")

  STAGE 2: TRAINING Serious vs Slight


Stage 2 trained in 28.9s | Best epoch: 1
Stage 2: Serious Recall=0.6641 | Macro F1=0.5024 | MCC=0.1505

  COMBINED HIERARCHICAL PIPELINE EVALUATION

Hierarchical Pipeline (threshold=0.09):
  Fatal Recall:   1.0000
  Serious Recall: 0.0992
  Slight Recall:  0.2921
  MCC:            0.0781
  Macro F1:       0.1980

  FULL COMPARISON — ALL APPROACHES
  Method                               Fatal R  Serious R  Slight R     MCC  Macro F1
  ------------------------------------------------------------------------------
  RF Baseline                           0.0000     0.2854    0.9590  0.0277    0.3088
  3-Class MLP (Step 5 winner)           0.5000     0.3550    0.5727  0.0987    0.3289
  Ensemble + Threshold (Step 7)         0.6250     0.3015    0.5735  0.1039    0.3248
  Hierarchical Pipeline (Steps 11-13)   1.0000     0.0992    0.2921  0.0781    0.1980
  ------------------------------------------------------------------------------

Confusion Matrix (Hierarchical):
                Pred Fat

### Stratified K-Fold Validation of Hierarchical Pipeline

We repeat the 5-Fold cross-validation for the hierarchical pipeline to confirm the improvement is real and stable, not just lucky on one split.

In [19]:
# ================================================================
# K-Fold Validation of Hierarchical Pipeline
# ================================================================

# Use full dataset with selected features
X_full_sel = np.concatenate([X_train_sel, X_val_sel], axis=0)
y_full = np.concatenate([y_train, y_val], axis=0)

K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
hier_fold_results = []

print(f"{'='*70}")
print(f"  HIERARCHICAL PIPELINE — STRATIFIED {K}-FOLD CROSS-VALIDATION")
print(f"{'='*70}")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_sel, y_full)):
    t0 = time.time()
    
    X_tr, y_tr = X_full_sel[train_idx], y_full[train_idx]
    X_vl, y_vl = X_full_sel[val_idx], y_full[val_idx]
    
    print(f"\n--- Fold {fold+1}/{K} ---")
    print(f"  Total: Train={len(y_tr)} | Val={len(y_vl)} | Fatal train={(y_tr==0).sum()} | Fatal val={(y_vl==0).sum()}")
    
    # --- Stage 1: Fatal vs Non-Fatal ---
    y_tr_s1 = (y_tr == 0).astype(np.int64)
    y_vl_s1 = (y_vl == 0).astype(np.int64)
    
    # SMOTE on training data
    sm = SMOTE(sampling_strategy=0.05, k_neighbors=3, random_state=SEED)
    X_tr_s1_sm, y_tr_s1_sm = sm.fit_resample(X_tr, y_tr_s1)
    
    # DataLoaders
    s1t = TensorDataset(torch.tensor(X_tr_s1_sm, dtype=torch.float32),
                         torch.tensor(y_tr_s1_sm, dtype=torch.long))
    # No WeightedRandomSampler — SMOTE already rebalanced; sampler would double-overcorrect
    s1_tl = DataLoader(s1t, batch_size=64, shuffle=True)
    s1v = TensorDataset(torch.tensor(X_vl, dtype=torch.float32),
                         torch.tensor(y_vl_s1, dtype=torch.long))
    s1_vl = DataLoader(s1v, batch_size=256, shuffle=False)
    
    # Train Stage 1
    m1 = BinaryMLP(X_tr.shape[1], [64, 32], dropout=0.3)
    m1.apply(init_weights)
    m1 = m1.to(device)
    c1 = FocalLoss(alpha=[1.0, 1.5], gamma=2.0).to(device)
    o1 = optim.AdamW(m1.parameters(), lr=1e-3, weight_decay=1e-4)
    ws1 = torch.optim.lr_scheduler.LinearLR(o1, start_factor=0.1, total_iters=5)
    cs1 = torch.optim.lr_scheduler.CosineAnnealingLR(o1, T_max=95, eta_min=1e-5)
    sc1 = torch.optim.lr_scheduler.SequentialLR(o1, schedulers=[ws1, cs1], milestones=[5])
    
    bf1, bs1, be1, pc1 = -1, None, 0, 0
    for ep in range(100):
        m1.train()
        for xb, yb in s1_tl:
            xb, yb = xb.to(device), yb.to(device)
            o1.zero_grad()
            loss = c1(m1(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m1.parameters(), max_norm=1.0)
            o1.step()
        v = evaluate_binary(m1, s1_vl, c1, device, pos_label=1)
        sc1.step()
        if v['recall'] >= 0.20 and v['f1'] > bf1:
            bf1 = v['f1']
            bs1 = copy.deepcopy(m1.state_dict())
            be1 = ep
            pc1 = 0
        else:
            pc1 += 1
        if pc1 >= 25:
            break
    if bs1 is not None:
        m1.load_state_dict(bs1)
    m1.eval()
    
    # Threshold calibration for Stage 1
    v1f = evaluate_binary(m1, s1_vl, c1, device, pos_label=1)
    bt1, br1 = 0.5, v1f['recall']
    for th in np.arange(0.05, 0.60, 0.01):
        p = (v1f['probs'] >= th).astype(int)
        r = recall_score(v1f['targets'], p, pos_label=1, zero_division=0)
        m = matthews_corrcoef(v1f['targets'], p)
        if r > br1 and m > 0.05:
            bt1, br1 = th, r
    
    # --- Stage 2: Serious vs Slight ---
    nf_tr = y_tr != 0
    nf_vl = y_vl != 0
    y_tr_s2 = (y_tr[nf_tr] - 1).astype(np.int64)
    y_vl_s2 = (y_vl[nf_vl] - 1).astype(np.int64)
    
    s2t = TensorDataset(torch.tensor(X_tr[nf_tr], dtype=torch.float32),
                         torch.tensor(y_tr_s2, dtype=torch.long))
    s2c = np.bincount(y_tr_s2, minlength=2)
    s2w = 1.0 / s2c[y_tr_s2]
    s2_tl = DataLoader(s2t, batch_size=64,
                       sampler=WeightedRandomSampler(torch.tensor(s2w, dtype=torch.float64),
                                                      len(y_tr_s2), replacement=True))
    s2v = TensorDataset(torch.tensor(X_vl[nf_vl], dtype=torch.float32),
                         torch.tensor(y_vl_s2, dtype=torch.long))
    s2_vl_loader = DataLoader(s2v, batch_size=256, shuffle=False)
    
    m2 = BinaryMLP(X_tr.shape[1], [64, 32], dropout=0.3)
    m2.apply(init_weights)
    m2 = m2.to(device)
    c2 = FocalLoss(alpha=[1.0, 1.0], gamma=1.0).to(device)
    o2 = optim.AdamW(m2.parameters(), lr=1e-3, weight_decay=1e-4)
    ws2 = torch.optim.lr_scheduler.LinearLR(o2, start_factor=0.1, total_iters=5)
    cs2 = torch.optim.lr_scheduler.CosineAnnealingLR(o2, T_max=95, eta_min=1e-5)
    sc2 = torch.optim.lr_scheduler.SequentialLR(o2, schedulers=[ws2, cs2], milestones=[5])
    
    bf2, bs2, be2, pc2 = -1, None, 0, 0
    for ep in range(100):
        m2.train()
        for xb, yb in s2_tl:
            xb, yb = xb.to(device), yb.to(device)
            o2.zero_grad()
            loss = c2(m2(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m2.parameters(), max_norm=1.0)
            o2.step()
        v2 = evaluate_binary(m2, s2_vl_loader, c2, device, pos_label=0)
        sc2.step()
        if v2['macro_f1'] > bf2:
            bf2 = v2['macro_f1']
            bs2 = copy.deepcopy(m2.state_dict())
            be2 = ep
            pc2 = 0
        else:
            pc2 += 1
        if pc2 >= 25:
            break
    if bs2 is not None:
        m2.load_state_dict(bs2)
    m2.eval()
    
    # Combined prediction
    hier_p = predict_hierarchical(X_vl, m1, m2, device, bt1)
    hr = recall_score(y_vl, hier_p, average=None, labels=[0, 1, 2], zero_division=0)
    hm = matthews_corrcoef(y_vl, hier_p)
    hf = f1_score(y_vl, hier_p, average='macro', zero_division=0)
    
    elapsed = time.time() - t0
    hier_fold_results.append({
        'fold': fold + 1, 'fatal_recall': hr[0], 'serious_recall': hr[1],
        'slight_recall': hr[2], 'mcc': hm, 'macro_f1': hf,
        'threshold': bt1, 'time': elapsed,
    })
    
    print(f"  Done in {elapsed:.1f}s | S1 threshold={bt1:.2f}")
    print(f"  Fatal R: {hr[0]:.4f} | Serious R: {hr[1]:.4f} | Slight R: {hr[2]:.4f} | "
          f"MCC: {hm:.4f} | Macro F1: {hf:.4f}")

# --- Summary ---
print(f"\n{'='*70}")
print(f"  HIERARCHICAL PIPELINE — {K}-FOLD SUMMARY")
print(f"{'='*70}")
print(f"  {'Fold':>4} {'Fatal R':>8} {'Serious R':>10} {'Slight R':>9} {'MCC':>7} {'Macro F1':>9}")
print(f"  {'-'*52}")

h_fr = [r['fatal_recall'] for r in hier_fold_results]
h_sr = [r['serious_recall'] for r in hier_fold_results]
h_slr = [r['slight_recall'] for r in hier_fold_results]
h_mcc = [r['mcc'] for r in hier_fold_results]
h_mf1 = [r['macro_f1'] for r in hier_fold_results]

for r in hier_fold_results:
    print(f"  {r['fold']:4d} {r['fatal_recall']:8.4f} {r['serious_recall']:10.4f} "
          f"{r['slight_recall']:9.4f} {r['mcc']:7.4f} {r['macro_f1']:9.4f}")

print(f"  {'-'*52}")
print(f"  {'Mean':>4} {np.mean(h_fr):8.4f} {np.mean(h_sr):10.4f} "
      f"{np.mean(h_slr):9.4f} {np.mean(h_mcc):7.4f} {np.mean(h_mf1):9.4f}")
print(f"  {'Std':>4} {np.std(h_fr):8.4f} {np.std(h_sr):10.4f} "
      f"{np.std(h_slr):9.4f} {np.std(h_mcc):7.4f} {np.std(h_mf1):9.4f}")

# Compare to previous K-Fold (3-class MLP)
print(f"\n  IMPROVEMENT over 3-Class MLP K-Fold:")
prev_fr, prev_mf1, prev_mcc = 0.5883, 0.3111, 0.0746
print(f"  Fatal Recall: {np.mean(h_fr):.4f} ± {np.std(h_fr):.4f}  (was {prev_fr:.4f})")
print(f"  Macro F1:     {np.mean(h_mf1):.4f} ± {np.std(h_mf1):.4f}  (was {prev_mf1:.4f})")
print(f"  MCC:          {np.mean(h_mcc):.4f} ± {np.std(h_mcc):.4f}  (was {prev_mcc:.4f})")

# Save
joblib.dump({
    'k': K,
    'fold_results': hier_fold_results,
    'summary': {
        'fatal_recall': {'mean': float(np.mean(h_fr)), 'std': float(np.std(h_fr))},
        'serious_recall': {'mean': float(np.mean(h_sr)), 'std': float(np.std(h_sr))},
        'slight_recall': {'mean': float(np.mean(h_slr)), 'std': float(np.std(h_slr))},
        'mcc': {'mean': float(np.mean(h_mcc)), 'std': float(np.std(h_mcc))},
        'macro_f1': {'mean': float(np.mean(h_mf1)), 'std': float(np.std(h_mf1))},
    },
}, '../outputs/models/hierarchical_kfold_results.joblib')
print(f"\nHierarchical K-Fold results saved.")

  HIERARCHICAL PIPELINE — STRATIFIED 5-FOLD CROSS-VALIDATION

--- Fold 1/5 ---
  Total: Train=14256 | Val=3565 | Fatal train=73 | Fatal val=19


  Done in 63.0s | S1 threshold=0.50
  Fatal R: 0.3158 | Serious R: 0.3771 | Slight R: 0.4387 | MCC: 0.1133 | Macro F1: 0.2996

--- Fold 2/5 ---
  Total: Train=14257 | Val=3564 | Fatal train=74 | Fatal val=18


  Done in 66.7s | S1 threshold=0.09
  Fatal R: 1.0000 | Serious R: 0.0337 | Slight R: 0.3442 | MCC: 0.0788 | Macro F1: 0.1911

--- Fold 3/5 ---
  Total: Train=14257 | Val=3564 | Fatal train=74 | Fatal val=18


  Done in 60.9s | S1 threshold=0.50
  Fatal R: 0.7222 | Serious R: 0.1734 | Slight R: 0.4346 | MCC: 0.0621 | Macro F1: 0.2621

--- Fold 4/5 ---
  Total: Train=14257 | Val=3564 | Fatal train=74 | Fatal val=18


  Done in 68.6s | S1 threshold=0.06
  Fatal R: 1.0000 | Serious R: 0.0354 | Slight R: 0.3462 | MCC: 0.0932 | Macro F1: 0.1937

--- Fold 5/5 ---
  Total: Train=14257 | Val=3564 | Fatal train=73 | Fatal val=19


  Done in 61.5s | S1 threshold=0.09
  Fatal R: 1.0000 | Serious R: 0.0455 | Slight R: 0.3161 | MCC: 0.0788 | Macro F1: 0.1861

  HIERARCHICAL PIPELINE — 5-FOLD SUMMARY
  Fold  Fatal R  Serious R  Slight R     MCC  Macro F1
  ----------------------------------------------------
     1   0.3158     0.3771    0.4387  0.1133    0.2996
     2   1.0000     0.0337    0.3442  0.0788    0.1911
     3   0.7222     0.1734    0.4346  0.0621    0.2621
     4   1.0000     0.0354    0.3462  0.0932    0.1937
     5   1.0000     0.0455    0.3161  0.0788    0.1861
  ----------------------------------------------------
  Mean   0.8076     0.1330    0.3759  0.0852    0.2265
   Std   0.2684     0.1329    0.0507  0.0171    0.0460

  IMPROVEMENT over 3-Class MLP K-Fold:
  Fatal Recall: 0.8076 ± 0.2684  (was 0.5883)
  Macro F1:     0.2265 ± 0.0460  (was 0.3111)
  MCC:          0.0852 ± 0.0171  (was 0.0746)

Hierarchical K-Fold results saved.
